# 03 - Rebuild Chroma DB dengan Schwartz-Hearst Expansion

**Tujuan**: re-embed semua 1.706 chunks dengan text yang sudah di-expand pakai
algoritma Schwartz-Hearst (handle akronim dengan internal letter seperti HBO,
VEGF, EGFR, dll), lalu simpan ke ChromaDB baru.

**Improvement over naive expansion**:
- Detect 673 acronyms (vs naive 425, +58%)
- Handle "internal letter" pattern: HBO -> Hyperbaric oxygenation
- Verified fix: PMID 7482275 (HBO case) RESULTS rank 371 -> 3

**Estimasi**:
- Cost API: ~$0.005 (1706 chunks x 150 tokens avg)
- Waktu: ~5 menit (batch 100)

**Input**: `notebooks/pubmedqa_bm25_sh.pkl` (sudah dibuat oleh build_sh_index.py)
**Output**: `notebooks/pubmedqa_chroma_sh/` (folder ChromaDB baru)


In [ ]:
import os, sys, time, pickle
from pathlib import Path
from dataclasses import dataclass

# # os.environ["OPENAI_API_KEY"] = "<REDACTED — set via shell env or .env file>"

from openai import OpenAI
import chromadb


@dataclass
class Document:
    text: str; pubid: str; question: str
    section_label: str; answer: str; decision: str

import __main__
__main__.Document = Document


HERE = Path(".").resolve()
NOTEBOOKS_DIR = HERE.parent if HERE.name == "BM25 Expansion" else HERE
BM25_SH_PATH = NOTEBOOKS_DIR / "pubmedqa_bm25_sh.pkl"
CHROMA_SH_PATH = NOTEBOOKS_DIR / "pubmedqa_chroma_sh"

print(f"BM25 SH path     : {BM25_SH_PATH} (exists: {BM25_SH_PATH.exists()})")
print(f"Chroma target    : {CHROMA_SH_PATH}")
print(f"Collection name  : pubmedqa_docs_sh")

assert BM25_SH_PATH.exists(), "BM25 SH index belum dibuat. Run: python build_sh_index.py"


In [ ]:
# Load expanded BM25 chunks
with open(BM25_SH_PATH, "rb") as f:
    saved = pickle.load(f)
documents = saved["documents"]
print(f"Loaded {len(documents)} SH-expanded chunks")
print()
print("Sample chunk PMID 7482275 RESULTS (HBO test case):")
for d in documents:
    if d.pubid == "7482275" and d.section_label == "RESULTS":
        print(d.text[:400])
        break


In [ ]:
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY belum di-set.")

client = OpenAI(api_key=api_key)
EMBED_MODEL = "text-embedding-3-small"

_emb = client.embeddings.create(model=EMBED_MODEL, input=["test"])
print(f"Embedding dim: {len(_emb.data[0].embedding)} (expected 1536)")


In [ ]:
def openai_embed(texts, model=EMBED_MODEL, max_retries=5):
    for attempt in range(max_retries):
        try:
            resp = client.embeddings.create(model=model, input=texts)
            return [d.embedding for d in resp.data]
        except Exception as e:
            err = str(e)
            if "429" in err or "rate" in err.lower():
                time.sleep((attempt + 1) * 10)
            elif "500" in err or "502" in err or "503" in err:
                time.sleep((attempt + 1) * 5)
            else:
                raise
    raise RuntimeError("OpenAI embeddings gagal")


chroma_client = chromadb.PersistentClient(path=str(CHROMA_SH_PATH))
COLL_NAME = "pubmedqa_docs_sh"

try:
    chroma_client.delete_collection(name=COLL_NAME)
    print(f"Deleted existing collection")
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLL_NAME, metadata={"hnsw:space": "cosine"}
)
print(f"Created collection \"{COLL_NAME}\"")

BATCH = 100
t0 = time.time()
total = len(documents)

for start in range(0, total, BATCH):
    batch = documents[start:start + BATCH]
    batch_texts = [d.text[:8000] for d in batch]
    batch_ids = [str(start + i) for i in range(len(batch))]
    batch_meta = [{"pubid": d.pubid, "section": d.section_label} for d in batch]

    embeddings = openai_embed(batch_texts)
    collection.add(
        ids=batch_ids, documents=batch_texts,
        metadatas=batch_meta, embeddings=embeddings,
    )
    done = start + len(batch)
    elapsed = time.time() - t0
    eta = elapsed / done * (total - done) / 60 if done < total else 0
    print(f"  [{done}/{total}] embedded | {elapsed:.0f}s elapsed | ETA {eta:.1f} mnt")

print(f"\nSelesai dalam {(time.time()-t0)/60:.1f} menit")
print(f"Total chunks di Chroma: {collection.count()}")


In [ ]:
# Verifikasi: query test untuk HBO case (idx 30)
test_query = "Necrotizing fasciitis: an indication for hyperbaric oxygenation therapy?"

qvec = openai_embed([test_query])[0]
results = collection.query(
    query_embeddings=[qvec], n_results=10,
    include=["distances", "metadatas"]
)

print(f"Query: {test_query}")
print(f"\nDense top-10 (Chroma SH):")
print(f"{'rank':>4} {'doc_id':>7} {'pubid':>10} {'section':<28} {'sim':>6}")
for rank, (did, dist, meta) in enumerate(zip(
    results["ids"][0], results["distances"][0], results["metadatas"][0]), 1):
    sim = 1 - dist
    src = " *SOURCE*" if meta["pubid"] == "7482275" else ""
    print(f"{rank:>4} {did:>7} {meta['pubid']:>10} {meta['section']:<28} {sim:.3f}{src}")


## Verifikasi sukses

Kalau **METHODS dan RESULTS dari PMID 7482275** (HBO case) muncul di top-10 Dense,
artinya re-embed berhasil dan dense retrieval sekarang juga terbantu oleh
Schwartz-Hearst expansion.

Lanjut ke notebook `04_baseline_openai_sh.ipynb` untuk full eksperimen.
